# 03 · Pipeline completa e submission

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FrancescoRomeo02/more-than-a-hack-2027-InkSight/blob/team-solution/solution/notebooks/03_pipeline_e_submission.ipynb)

1. Feature dal tracking → punteggi per frame per H/A/D (+ Smart Tagging come prior)
2. **HMM a durata minima** decodificato con Viterbi, per tempo: niente possessi di 3 frame, cambi di stato penalizzati
3. Scrittura del CSV, validazione, controllo di riproducibilità

Parametri: `params.json` scritto dal notebook 02 (su Colab: `DRIVE_DIR/params.json`). **Se manca il notebook si ferma**: niente ripiego silenzioso sui default.

In [ ]:
# --- Setup: funziona sia in locale sia su Google Colab ----------------------------
# Su Colab: monta Drive, clona la repo in /content/mtah e legge i dati da DRIVE_DIR/data.
# params.json e la cache stanno direttamente in DRIVE_DIR; il CSV della submission viene copiato li'.
import os, sys, time, shutil, subprocess
REPO_URL  = "https://github.com/FrancescoRomeo02/more-than-a-hack-2027-InkSight.git"
BRANCH    = "team-solution"
DRIVE_DIR = "/content/drive/MyDrive/mtah2027"        # <- cartella su Drive con dentro data/tactical/...
IN_COLAB  = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = "/content/mtah"
    if not os.path.exists(os.path.join(ROOT, "games.py")):
        subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO_URL, ROOT], check=True)
    else:
        subprocess.run(["git", "-C", ROOT, "pull", "-q"], check=False)
    os.chdir(ROOT)
    # Versioni fissate (solution/requirements.txt): il 02 salva deadball_model.pkl e il 03 lo
    # ricarica, quindi scikit-learn (e numpy) devono essere IDENTICI nei due runtime.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    os.path.join(ROOT, "solution", "requirements.txt")], check=True)
    from importlib.metadata import version as _v
    _stale = [m for m in ("numpy", "pandas", "scipy", "sklearn", "matplotlib")
              if m in sys.modules and getattr(sys.modules[m], "__version__", None)
              != _v({"sklearn": "scikit-learn"}.get(m, m))]
    if _stale:
        print("pip ha cambiato librerie gia' caricate", _stale,
              "-> riavvio il runtime: poi riesegui il notebook dall'inizio (la seconda volta non succede)")
        os.kill(os.getpid(), 9)
    # Tutto cio' che deve passare da un notebook all'altro sta su Drive (ogni notebook ha il suo runtime):
    os.environ.setdefault("MTAH_DATA_DIR", os.path.join(DRIVE_DIR, "data"))           # dati della challenge
    os.environ.setdefault("MTAH_CACHE_DIR", os.path.join(DRIVE_DIR, "cache"))         # ATD parsato, Metrica, log calibrazione
    os.environ.setdefault("MTAH_PARAMS", os.path.join(DRIVE_DIR, "params.json"))      # scritto da 02, letto da 03
    # (le variabili vanno impostate PRIMA di importare `possession`)
    if not os.path.isdir(os.path.join(os.environ["MTAH_DATA_DIR"], "tactical")):
        print("ATTENZIONE: dati non trovati in", os.environ["MTAH_DATA_DIR"], "- carica la cartella data/ su Drive")
    p = os.environ["MTAH_PARAMS"]
    print("params.json su Drive:", time.ctime(os.path.getmtime(p)) if os.path.exists(p) else "NON ANCORA PRESENTE (esegui il notebook 02)")
else:
    ROOT = os.environ.get("MTAH_ROOT") or os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
    if not os.path.exists(os.path.join(ROOT, "games.py")):      # notebook lanciato dalla root
        ROOT = os.getcwd()

def save_to_drive(path):
    """Su Colab copia un file in DRIVE_DIR, cosi' non si perde quando il runtime si resetta."""
    if IN_COLAB:
        os.makedirs(DRIVE_DIR, exist_ok=True)
        shutil.copy(path, DRIVE_DIR)
        print("copiato su Drive:", os.path.join(DRIVE_DIR, os.path.basename(path)))

sys.path[:0] = [ROOT, os.path.join(ROOT, "solution"), os.path.join(ROOT, "evaluation")]
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore", category=RuntimeWarning)
print("ROOT =", ROOT, "| Colab" if IN_COLAB else "| locale")

In [ ]:
import json
from possession import pipeline, data
TEAM_NAME = "team_name"        # <- il nome della squadra: il file deve chiamarsi <team_name>.csv
PARAMS = pipeline.params_path()
params, cfg = pipeline.load_params(PARAMS, required=True)     # errore chiaro se 02 non e' stato eseguito
meta = json.load(open(PARAMS))
print("parametri da:", PARAMS, "|", time.ctime(os.path.getmtime(PARAMS)))
print("calibrazione:", meta.get("n_trials"), "configurazioni, spellF1 Metrica", meta.get("spell_f1_metrica"))
print("decoding:", {"min_len": cfg["min_len"], "switch": next(iter(cfg["switch_cost"].values()))})
t = time.time(); res = pipeline.run(params, cfg); print(f"{time.time()-t:.1f}s")
labels = res["labels"]

In [ ]:
# Struttura dei possessi previsti (si guarda il NOSTRO output, non la ground truth)
for half, a, b in data.graded_blocks():
    s = labels[a:b+1]; edges = np.flatnonzero(s[1:] != s[:-1]) + 1
    segs = np.split(s, edges); pos = [x for x in segs if x[0] in "HA"]
    print(f"tempo {half}: {len(pos)} possessi, durata media {np.mean([len(x) for x in pos])/25:.1f}s, "
          f"pause D: {sum(x[0]=='D' for x in segs)}")

In [ ]:
# Una finestra per vedere punteggi e decisioni
a, b = 30000, 33000; x = np.arange(a, b)/25; ll = res["scores"]
fig, ax = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
ax[0].plot(x, ll[a:b, 0]-ll[a:b, 1], lw=.7, label="evidenza H - A"); ax[0].plot(x, ll[a:b, 2], lw=.7, label="evidenza D"); ax[0].legend()
ax[1].plot(x, [{"H":0,"A":1,"D":2,"X":np.nan}[c] for c in labels[a:b]]); ax[1].set_yticks([0,1,2], ["H","A","D"]); ax[1].set_xlabel("s")
plt.show()

In [ ]:
out = os.path.join(ROOT, "evaluation", "results", f"{TEAM_NAME}.csv")
pipeline.write_submission(labels, out)
save_to_drive(out)
import subprocess
print(subprocess.run([sys.executable, os.path.join(ROOT, "evaluation", "validate_submission.py"), out],
                     capture_output=True, text=True).stdout)

## Controllo di correttezza sui 5 minuti pubblici

⚠️ **Solo** per verificare che la pipeline sia corretta (frame allineati, etichette sensate). Il regolamento vieta di usarli per scegliere tra versioni o tarare parametri: 17 possessi non bastano e portano a scelte sbagliate.

In [ ]:
# Classifica completa + dettaglio della NOSTRA submission (prima l'output veniva tagliato a 3000 caratteri).
# --bootstrap 1000: intervallo di confidenza, con 17 possessi il rumore e' grande. Report completi in evaluation/reports/.
out_eval = subprocess.run([sys.executable, os.path.join(ROOT, "evaluation", "run_evaluation.py"), "--bootstrap", "1000"],
                          capture_output=True, text=True, cwd=os.path.join(ROOT, "evaluation"))
print(out_eval.stderr[-2000:])
lines = out_eval.stdout.splitlines()
names = {f[:-4] for f in os.listdir(os.path.join(ROOT, "evaluation", "results")) if f.endswith(".csv")}
starts = [i for i, l in enumerate(lines) if l.strip() in names and l == l.strip()]   # inizio dei blocchi di dettaglio
print("\n".join(lines[:starts[0]] if starts else lines))                              # tabella e note
for k, i in enumerate(starts):
    if lines[i].strip() == TEAM_NAME:
        print("\n".join(lines[i:starts[k + 1] if k + 1 < len(starts) else len(lines)]))

## Riproducibilità: due esecuzioni devono dare lo stesso file

In [ ]:
import hashlib
h1 = hashlib.sha256(open(out, "rb").read()).hexdigest()
res2 = pipeline.run(params, cfg, verbose=False); tmp = out + ".check"; pipeline.write_submission(res2["labels"], tmp)
h2 = hashlib.sha256(open(tmp, "rb").read()).hexdigest(); os.remove(tmp)
print("identico" if h1 == h2 else "DIVERSO!", h1[:16])